# PRNU Group A on Google Colab

Use a **GPU runtime** (Runtime → Change runtime type → T4 or better).

**Data:** Dresden Image Database from Kaggle (`micscodes/dresden-image-database`). You need a [Kaggle API token](https://www.kaggle.com/settings) (`kaggle.json`).

In [ ]:
# GPU check
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Get the code

**Option A — Git clone** (replace with your fork or team repo URL):

```python
!git clone https://github.com/YOUR_ORG/team-project-machinelearning.git
%cd team-project-machinelearning/prnu_project
```

**Option B — Upload** `prnu_project` as a zip to Colab Files, unzip, then `%cd` into `prnu_project`.

The cells below assume you end up with your cwd = `prnu_project` (folder that contains `experiments/`, `src/`, `configs/`).

In [ ]:
# --- EDIT: set REPO_URL, or put an unzipped `prnu_project` folder in /content ---
REPO_URL = "https://github.com/YOUR_ORG/team-project-machinelearning.git"  # change me

import os
from pathlib import Path

root = Path("/content")
if (root / "prnu_project" / "experiments").is_dir():
    %cd /content/prnu_project
elif (root / "repo_clone" / "prnu_project" / "experiments").is_dir():
    %cd /content/repo_clone/prnu_project
else:
    !git clone {REPO_URL} repo_clone
    %cd /content/repo_clone/prnu_project

print("cwd:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 2. Kaggle credentials

Upload `kaggle.json` once:
- **From Drive:** mount Drive and copy `kaggle.json` to `~/.kaggle/kaggle.json`, or
- **Manual:** use Colab’s file upload for `kaggle.json`.

In [ ]:
import json
import os
from pathlib import Path

kg = Path.home() / ".kaggle"
kg.mkdir(parents=True, exist_ok=True)

# Uncomment ONE of the following:

# --- From Google Drive (path to your kaggle.json) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy("/content/drive/MyDrive/kaggle.json", kg / "kaggle.json")

# --- Or upload in Colab: Files pane → upload kaggle.json to /content ---
# import shutil
# shutil.copy("/content/kaggle.json", kg / "kaggle.json")

os.chmod(kg / "kaggle.json", 0o600)
print("kaggle.json ready:", (kg / "kaggle.json").exists())

In [ ]:
%%bash
mkdir -p data/raw/dresden
kaggle datasets download -d micscodes/dresden-image-database -p data/raw/dresden --unzip
ls -la data/raw/dresden | head

## 3. Train Group A (GPU)

- Full-quality training: `configs/default.yaml` (30 epochs CNN/Siamese; still faster on GPU than laptop CPU).
- Quick end-to-end sanity check: `configs/smoke.yaml` (2 epochs).

Results are written to `results/group_A.json`.

If DataLoader workers hang on Colab, run from a terminal with `PYTHONWARNINGS=ignore` or temporarily change `num_workers=0` in `experiments/run_group_A.py` — Colab sometimes prefers `num_workers=0`.

In [ ]:
!python experiments/run_group_A.py --config configs/default.yaml --device cuda

In [ ]:
# Quick smoke run (optional)
# !python experiments/run_group_A.py --config configs/smoke.yaml --device cuda

In [ ]:
# Show metrics preview
from pathlib import Path
import json
p = Path("results/group_A.json")
if p.exists():
    d = json.loads(p.read_text())
    for block in ["A1", "A2", "A3"]:
        print(block, json.dumps(d.get(block, {}), indent=2)[:1200], "...")
else:
    print("No results yet — run training cell above.")

## 4. Save results

Download `results/group_A.json` from the Files pane, or copy to Drive:

```python
# from google.colab import drive
# drive.mount("/content/drive")
# !cp results/group_A.json /content/drive/MyDrive/
```